# AGL — RoBERTa Fine-Tuning for Prompt Classification

**Adaptive Guardrail Layer (AGL) — Multi-Stage Neural Classifier for LLM Security**

This notebook fine-tunes `roberta-base` on the AGL 4-class dataset (Benign, Injection, Jailbreak, Exfiltration) using a T4 GPU on Google Colab.

**Runtime:** GPU → T4 (Colab free tier) or better

In [ ]:
# ── 1. Setup ────────────────────────────────────────────────────────────────
# Clone/update the repo

import os, sys
REPO_DIR = '/content/agl-capstone'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/joeldiev/aai-590-group-8-capstone.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── 2. Install dependencies ─────────────────────────────────────────────────
!pip install -r requirements.txt -q
!pip install wandb -q  # optional: experiment tracking

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 3: Build Dataset

Run the data pipeline to download, merge, label, deduplicate, balance, and split the dataset.

- `--phase mvp` → 3-class (Benign, Injection, Jailbreak) using deepset + JailBreakV-28K only
- `--phase full` → 4-class (adds Exfiltration) with all sources + synthetic data

In [ ]:
# ── 3. Build dataset ────────────────────────────────────────────────────────
# Change to "full" once all datasets and synthetic exfiltration are ready
!python -m src.run --stage data --phase full

In [ ]:
# ── 3b. Verify dataset ──────────────────────────────────────────────────────
import pandas as pd

for split in ['train', 'val', 'test']:
    df = pd.read_parquet(f'data/processed/{split}.parquet')
    print(f"\n{split}: {len(df)} samples")
    print(df['unified_label'].value_counts().to_string())

## Step 4: Fine-Tune RoBERTa

Train the classifier with class-weighted cross-entropy, early stopping, and linear LR schedule.

Default hyperparameters (from `src/config.py`):
- LR: 2e-5, warmup: 10%, weight decay: 0.01
- Batch: 16 (gradient accum 2 → effective 32)
- Max epochs: 5, early stopping patience: 2
- Max sequence length: 256

In [ ]:
# ── 4. Train RoBERTa classifier ─────────────────────────────────────────────
# Default hyperparameters — override with --lr and --max-length flags
!python -m src.run --stage train --mode classifier

## Step 4b: Hyperparameter Experiments (Optional)

Run multiple experiments varying LR and max sequence length. Compare results to pick the best config.

In [ ]:
# ── 4b. Hyperparameter sweep (optional) ─────────────────────────────────────
# Results are saved to separate output directories.

import itertools
from src.training.train import train_classifier
from src.config import MODELS_DIR

experiments = {
    'lr': [1e-5, 2e-5, 3e-5],
    'max_length': [128, 256],
}

results = []
for lr, max_len in itertools.product(experiments['lr'], experiments['max_length']):
    name = f"lr{lr}_ml{max_len}"
    print(f"\n{'='*60}")
    print(f"Experiment: {name}")
    print(f"{'='*60}")
    out = MODELS_DIR / f"classifier_{name}"
    try:
        best_path = train_classifier(
            output_dir=out,
            learning_rate=lr,
            max_length=max_len,
        )
        results.append({'lr': lr, 'max_length': max_len, 'path': str(best_path)})
    except Exception as e:
        print(f"  FAILED: {e}")
        results.append({'lr': lr, 'max_length': max_len, 'path': None, 'error': str(e)})

import pandas as pd
pd.DataFrame(results)

## Step 5: Fit Anomaly Detector

Extract [CLS] embeddings from the fine-tuned model and fit the Mahalanobis OOD detector.
Threshold is calibrated on the validation set (95% in-distribution recall).

In [ ]:
# ── 5. Fit Mahalanobis anomaly detector ─────────────────────────────────────
!python -m src.run --stage train --mode anomaly

## Step 6: Evaluate

Run the full evaluation suite: keyword baseline, TF-IDF/SVM baseline, RoBERTa classifier, and (if trained) RoBERTa + anomaly detector.

In [ ]:
# ── 6. Run evaluation ───────────────────────────────────────────────────────
!python -m src.run --stage evaluate

## Step 7: Visualizations

Generate publication-quality figures: confusion matrix, F1 comparison, ROC curves, latency distribution.

In [ ]:
# ── 7. Generate visualizations ──────────────────────────────────────────────
import numpy as np
import pandas as pd
from src.config import RESULTS_DIR, FIGURES_DIR, PROCESSED_DIR, MODELS_DIR
from src.evaluation.visualizations import plot_confusion_matrix, plot_f1_comparison
from src.evaluation.metrics import benchmark_latency
from src.evaluation.baselines import keyword_blocklist_baseline, tfidf_svm_baseline
from src.models.agl_pipeline import AGLPipeline

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load test data
test_df = pd.read_parquet(PROCESSED_DIR / 'test.parquet')
train_df = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
y_true = test_df['label'].values

# Baselines
kw_preds = keyword_blocklist_baseline(test_df)
svm_preds, _ = tfidf_svm_baseline(train_df, test_df)

# RoBERTa predictions
checkpoint = MODELS_DIR / 'classifier' / 'best'
pipeline = AGLPipeline.from_checkpoint(checkpoint)
roberta_preds = np.array([pipeline.predict(t).label_id for t in test_df['text']])

# Confusion matrix
plot_confusion_matrix(y_true, roberta_preds, title='AGL RoBERTa', save_path=FIGURES_DIR / 'confusion_matrix.png')

# F1 comparison
from sklearn.metrics import f1_score
f1_results = {
    'Keyword Blocklist': f1_score(y_true, kw_preds, average='macro', zero_division=0),
    'TF-IDF + SVM': f1_score(y_true, svm_preds, average='macro', zero_division=0),
    'RoBERTa (AGL)': f1_score(y_true, roberta_preds, average='macro', zero_division=0),
}
plot_f1_comparison(f1_results, save_path=FIGURES_DIR / 'f1_comparison.png')

# Latency
latency = benchmark_latency(pipeline, test_df['text'].head(50).tolist(), n_runs=2)
print(f"Latency: mean={latency['mean_ms']:.1f}ms, p99={latency['p99_ms']:.1f}ms")
print('\nAll figures saved to:', FIGURES_DIR)

## Step 8: Save Checkpoint to Drive

Copy the trained model and results to Google Drive for persistence across Colab sessions.

In [ ]:
# ── 8. Save to Google Drive ─────────────────────────────────────────────────
import shutil

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/AGL_Capstone'
os.makedirs(DRIVE_DIR, exist_ok=True)

for src, dst_name in [
    ('models/classifier/best', 'classifier_best'),
    ('models/anomaly', 'anomaly'),
    ('results', 'results'),
    ('data/processed', 'data_processed'),
]:
    if os.path.exists(src):
        dst = f'{DRIVE_DIR}/{dst_name}'
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Saved {src} -> {dst}')

print('\nDone! All artifacts saved to Google Drive.')